# Test Notebooks to see if we can load some llms

In [1]:
%pip install -qU langchain-openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts.chat import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)
from langchain_openai import ChatOpenAI

In [ ]:
inference_server_url = "https://user-jonasmorin-844854-vllm-user.lab.sspcloud.fr/v1/"

llm = ChatOpenAI(
    model="/root/.cache/huggingface/Phi-3.5-mini-instruct",
    openai_api_key="EMPTY",
    openai_api_base=inference_server_url,
    max_tokens=5,
    temperature=0,
)

In [ ]:
messages = [
    SystemMessage(
        content="You are a helpful assistant that translates English to Italian."
    ),
    HumanMessage(
        content="Translate the following sentence from English to Italian: I love programming."
    ),
]
llm.invoke(messages)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate(
    [
        (
            "system",
            "You are a helpful assistant that translates {input_language} to {output_language}.",
        ),
        (   "human", 
            "{input}"
        ),
    ]
)

chain = prompt | llm
chain.invoke(
    {
        "input_language": "English",
        "output_language": "German",
        "input": "I love programming.",
    }
)

In [6]:
answer = chain.invoke(
    {
        "input_language": "English",
        "output_language": "German",
        "input": "I love programming.",
    }
)

In [14]:
print(answer)
print(answer.content)
print(answer.response_metadata['token_usage']['total_tokens'])

content=' Ich liebe Programm' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 21, 'total_tokens': 26, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': '/root/.cache/huggingface/Phi-3.5-mini-instruct', 'system_fingerprint': None, 'finish_reason': 'length', 'logprobs': None} id='run-10f795dc-bad3-420d-a9e7-6f587a2ec45b-0' usage_metadata={'input_tokens': 21, 'output_tokens': 5, 'total_tokens': 26, 'input_token_details': {}, 'output_token_details': {}}
 Ich liebe Programm
26


In [19]:
from langchain_core.callbacks.base import BaseCallbackHandler
from typing import Dict, List, Any

class CustomHandler(BaseCallbackHandler):
    def on_llm_start(
        self, serialized: Dict[str, Any], prompts: List[str], **kwargs: Any
    ) -> Any:
        formatted_prompts = "\n".join(prompts)
        _log.info(f"Prompt:\n{formatted_prompts}")


output = chain.invoke({
        "input_language": "English",
        "output_language": "German",
        "input": "I love programming.",
    }, config={"callbacks": [CustomHandler()]})

Error in CustomHandler.on_llm_start callback: NameError("name '_log' is not defined")


In [22]:
prompt_as_string = prompt.format(
        input_language = "English",
        output_language =  "German",
        input =  "I love programming.",
)
print(prompt_as_string)

System: You are a helpful assistant that translates English to German.
Human: I love programming.


In [13]:
%pip install langchain
%pip install -U langchain-community
%pip install sentence-transformers
%pip install chromadb

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.8 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 88.2 MB/s eta 0:00:00
  Created wheel for pypika: filename=pypika-0.48.9-py2.py3-none-any.whl size=53800 sha256=2d9d03ecb660bef889cbd21aec173c204ab0ab115b8706e27c2155459cec2e26
  Stored in directory: /home/onyxia/.cache/pip/wheels/d5/3d/69/8d68d249cd3de2584f226e27fd431d6344f7d70fd856ebd01

In [1]:
import pandas as pd
from langchain.prompts.example_selector import SemanticSimilarityExampleSelector
from langchain.vectorstores.chroma import Chroma
from langchain.embeddings import HuggingFaceEmbeddings


question = "How many babies in geneva in 2020 ?"

table_name = "baby_names_favorite_firstname"

max_shot = 2

# def generate_sql_in_context_learning_similar_shots(question, table_name, n_shots = 2):
file_path = "../api/data/query_questions_db.csv"

# find the n_shots closest questions from the query_questions_db and the table
with open(file_path) as f:
    origin_of_shots = pd.read_csv(f, delimiter= ',')

#print(origin_of_shots)

examples = origin_of_shots.loc[origin_of_shots['db_id']==table_name]
examples = examples.reset_index()
few_shot_examples = []
meta_data = []

for j in range(len(examples)):
    ex_question = examples.loc[j,'question'].replace("\n","").strip()
    ex_query = examples.loc[j,'query']
    few_shot_examples.append({"question":ex_question})
    meta_data.append({"question":ex_question,"query":ex_query})

print(meta_data)

to_vectorize = [" ".join(example.values()) for example in few_shot_examples]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/distiluse-base-multilingual-cased-v2")

vectorstore = None
vectorstore = Chroma.from_texts(to_vectorize, embeddings, metadatas=meta_data)

# Lower score is more similar
answers = vectorstore.similarity_search_with_score(question, max_shot)

print(answers)

for item in answers:
    print(item[0].metadata['question'])
    

# # print (f'######{question}###########')
# nl2sql_pairs=[]
# scores=[]
# for item in answers:
#     print(item[0].metadata['question']) # print out score          
#     # print(item[0].metadata['query']) 
#     # print(item[0].metadata['rules']) 
#     nl=item[0].metadata['question']
#     q=item[0].metadata['query']
#     score=item[1]
#     scores.append(score)
#     nl2sql_pairs.append(
#         {'question':nl, 'query':q,'score':score}
#     )
# # print('###########') 

# examples_selector = SemanticSimilarityExampleSelector(
#     vectorstore=vectorstore,
#     k = max_shot,
# )
# examples_prompt = PromptTemplate(
#     input_variables=["question","query"],
#     template="### Question\n{question}\n### SQL query\n{query}",
# )  


[{'question': 'How many girls borned in 2014 in Switzeland is named as Lena. Return me the rank as well.', 'query': "SELECT amount, rank\nFROM baby_names_favorite_firstname as bnff\nJOIN spatial_unit as su ON bnff.spatialunit_uid = su.spatialunit_uid\nWHERE year = 2014\n    AND bnff.gender = 'girl'\n    AND bnff.first_name = 'Lena'\n    AND su.name = 'Switzerland'\n    AND su.country = 'TRUE';\n"}, {'question': 'Show me the most popular names of girls in Switzerland in 2016.', 'query': "SELECT first_name, gender, amount, rank\nFROM baby_names_favorite_firstname as bnff\nJOIN spatial_unit as su ON bnff.spatialunit_uid = su.spatialunit_uid\nWHERE year = 2016 and gender = 'girl' AND su.country = ''True'' AND su.name = 'Switzerland'\nORDER BY bnff.amount DESC LIMIT 1;\n"}, {'question': 'How many boys were registered in Canton Valais in year 2011?', 'query': "SELECT sum(amount)\nFROM baby_names_favorite_firstname as bnff\nJOIN spatial_unit as su ON bnff.spatialunit_uid = su.spatialunit_uid\

/tmp/ipykernel_46859/391890768.py:37: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/distiluse-base-multilingual-cased-v2")
/opt/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[(Document(metadata={'query': 'SELECT DISTINCT COUNT(bnff.first_name)\nFROM baby_names_favorite_firstname as bnff;\n', 'question': 'How many baby names were registered?'}, page_content='How many baby names were registered?'), 0.7536030411720276), (Document(metadata={'query': 'SELECT DISTINCT COUNT(bnff.first_name)\nFROM baby_names_favorite_firstname as bnff\nWHERE bnff.year = 2011;\n', 'question': 'How many favorite babynames are registered in year 2011?'}, page_content='How many favorite babynames are registered in year 2011?'), 0.7887356877326965)]
How many baby names were registered?
How many favorite babynames are registered in year 2011?


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

question = ["original text"]
documents = ["original text", "another text", "another other text"]

question_transformed = TfidfVectorizer().fit_transform(question)
tfidf = TfidfVectorizer().fit_transform(documents)
# no need to normalize, since Vectorizer will return normalized tf-idf
# pairwise_similarity = tfidf * tfidf.T
print(question_transformed)
print(tfidf)

pairwise_similarity = tfidf * question_transformed

print(pairwise_similarity)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2 stored elements and shape (1, 2)>
  Coords	Values
  (0, 0)	0.7071067811865475
  (0, 1)	0.7071067811865475
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 7 stored elements and shape (3, 4)>
  Coords	Values
  (0, 1)	0.8610369959439764
  (0, 3)	0.5085423203783267
  (1, 3)	0.6133555370249717
  (1, 0)	0.7898069290660905
  (2, 3)	0.4254405389711991
  (2, 0)	0.5478321549274363
  (2, 2)	0.7203334490549893


ValueError: matmul: dimension mismatch with signature (n,k=4),(k=1,m)->(n,m)